# Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)
import os

os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

# Sesión de Spark

In [ ]:
spark = (
    SparkSession.builder
    .appName("Lab7_ENEIC")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

# Setup

In [ ]:
archivos = {
    "2025T1": "datos/Personas_ENEIC_T1_2025.xlsx",
    "2025T2": "datos/Personas-ENEIC-T2-2025.xlsx",
    "2025T3": "datos/Base-de-datos-Personas-ENEIC-III-2025.xlsx",
    "2025T4": "datos/Base-de-datos-Personas-ENEIC-IV-2025.xlsx",
    "2026T1": "datos/Base-de-datos-Personas-ENEIC-I-2026.xlsx"
}

ruta_diccionario = "datos/Diccionario-Personas-ENEIC-I-2026.xlsx"

columnas = [
    "P05D01",       # salario mensual
    "P02A03",       # edad
    "P05C07A",      # antigüedad - años
    "P05C07B",      # antigüedad - meses
    "P05H01A",      # horas semanales
    "P03A03A",      # nivel educativo
    "P05C16",       # categoría ocupacional
    "DOMINIO",
    "OCUPADOS",
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR",
    "ANIO",
    "TRIMESTRE"
]

ruta_t1_2025 = archivos["2025T1"]

excel_t1 = pd.ExcelFile(ruta_t1_2025)

print(excel_t1.sheet_names)

t1_2025_pd = pd.read_excel(
    ruta_t1_2025
)
print("Dimensiones:", t1_2025_pd.shape)

renombres = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado"
}

In [ ]:
def cargar_periodo(ruta, periodo, anio, trimestre):

    # Leer Excel
    df = pd.read_excel(ruta)

    # Guardar cantidad original
    n_original = len(df)

    # Seleccionar únicamente las columnas necesarias
    df = df[columnas].copy()

    # Renombrar variables analíticas
    df = df.rename(columns=renombres)

    # Agregar trazabilidad del archivo
    df["archivo_origen"] = ruta
    df["periodo_archivo"] = periodo
    df["anio_archivo"] = anio
    df["trimestre_calendario"] = trimestre

    # Homologar variables numéricas
    columnas_numericas = [
        "salario_mensual",
        "edad",
        "antiguedad_anios",
        "antiguedad_meses",
        "horas_semanales",
        "FACTOR"
    ]

    for col in columnas_numericas:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # Homologar códigos categóricos como strings
    columnas_categoricas = [
        "nivel_educativo",
        "categoria_ocupacional",
        "dominio",
        "ocupado"
    ]

    for col in columnas_categoricas:
        df[col] = (
            pd.to_numeric(df[col], errors="coerce")
            .astype("Int64")
            .astype("string")
        )

    # IDs para auditoría
    df["NUM_HOGAR"] = (
        df["NUM_HOGAR"]
        .astype("Int64")
        .astype("string")
    )

    df["NUM_PERSONA"] = (
        df["NUM_PERSONA"]
        .astype("Int64")
        .astype("string")
    )

    # Construir antigüedad en años
    df["antiguedad"] = (
        df["antiguedad_anios"]
        + df["antiguedad_meses"] / 12
    )

    return df, n_original

In [ ]:
t1_2025, n_t1_2025 = cargar_periodo(
    archivos["2025T1"],
    "2025T1",
    2025,
    1
)

t2_2025, n_t2_2025 = cargar_periodo(
    archivos["2025T2"],
    "2025T2",
    2025,
    2
)

t3_2025, n_t3_2025 = cargar_periodo(
    archivos["2025T3"],
    "2025T3",
    2025,
    3
)

t4_2025, n_t4_2025 = cargar_periodo(
    archivos["2025T4"],
    "2025T4",
    2025,
    4
)

t1_2026, n_t1_2026 = cargar_periodo(
    archivos["2026T1"],
    "2026T1",
    2026,
    1
)

In [ ]:
schema_eneic = StructType([
    StructField("salario_mensual", DoubleType(), True),
    StructField("edad", DoubleType(), True),
    StructField("antiguedad_anios", DoubleType(), True),
    StructField("antiguedad_meses", DoubleType(), True),
    StructField("horas_semanales", DoubleType(), True),

    StructField("nivel_educativo", StringType(), True),
    StructField("categoria_ocupacional", StringType(), True),
    StructField("dominio", StringType(), True),
    StructField("ocupado", StringType(), True),

    StructField("NUM_HOGAR", StringType(), True),
    StructField("NUM_PERSONA", StringType(), True),

    StructField("FACTOR", DoubleType(), True),

    StructField("ANIO", IntegerType(), True),
    StructField("TRIMESTRE", IntegerType(), True),

    StructField("archivo_origen", StringType(), True),
    StructField("periodo_archivo", StringType(), True),

    StructField("anio_archivo", IntegerType(), True),
    StructField("trimestre_calendario", IntegerType(), True),

    StructField("antiguedad", DoubleType(), True)
])

In [ ]:
def preparar_para_spark(df):

    df = df.copy()

    # Variables continuas
    columnas_double = [
        "salario_mensual",
        "edad",
        "antiguedad_anios",
        "antiguedad_meses",
        "horas_semanales",
        "FACTOR",
        "antiguedad"
    ]

    for col in columnas_double:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].apply(
            lambda x: float(x) if pd.notna(x) else None
        )

    # Variables enteras
    columnas_int = [
        "ANIO",
        "TRIMESTRE",
        "anio_archivo",
        "trimestre_calendario"
    ]

    for col in columnas_int:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].apply(
            lambda x: int(x) if pd.notna(x) else None
        )

    # Variables de texto
    columnas_string = [
        "nivel_educativo",
        "categoria_ocupacional",
        "dominio",
        "ocupado",
        "NUM_HOGAR",
        "NUM_PERSONA",
        "archivo_origen",
        "periodo_archivo"
    ]

    for col in columnas_string:
        df[col] = df[col].apply(
            lambda x: str(x) if pd.notna(x) else None
        )

    return df

In [ ]:
spark_t1_2025 = spark.createDataFrame(
    preparar_para_spark(t1_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t2_2025 = spark.createDataFrame(
    preparar_para_spark(t2_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t3_2025 = spark.createDataFrame(
    preparar_para_spark(t3_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t4_2025 = spark.createDataFrame(
    preparar_para_spark(t4_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t1_2026 = spark.createDataFrame(
    preparar_para_spark(t1_2026).to_dict("records"),
    schema=schema_eneic
)

In [ ]:
datos_2025 = (
    spark_t1_2025
    .unionByName(spark_t2_2025)
    .unionByName(spark_t3_2025)
    .unionByName(spark_t4_2025)
)



datos_2025.groupBy(
    "periodo_archivo"
).count().orderBy(
    "periodo_archivo"
).show()

In [ ]:
variables_analisis = [
    "salario_mensual",
    "edad",
    "antiguedad_anios",
    "antiguedad_meses",
    "antiguedad",
    "horas_semanales",
    "nivel_educativo",
    "categoria_ocupacional",
    "dominio",
    "ocupado",
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR"
]

total_2025 = datos_2025.count()

faltantes = []

for col in variables_analisis:
    n_faltantes = datos_2025.filter(
        F.col(col).isNull() |
        F.isnan(F.col(col)) if dict(datos_2025.dtypes)[col] in ["double", "float"]
        else F.col(col).isNull()
    ).count()

    faltantes.append(
        (
            col,
            n_faltantes,
            round(n_faltantes / total_2025 * 100, 2)
        )
    )

faltantes_df = spark.createDataFrame(
    faltantes,
    ["variable", "faltantes", "porcentaje"]
)

faltantes_df.show(truncate=False)

In [ ]:
duplicados_clave = (
    datos_2025
    .groupBy(
        "periodo_archivo",
        "NUM_HOGAR",
        "NUM_PERSONA"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Claves duplicadas:",
    duplicados_clave.count()
)

duplicados_clave.show(10, truncate=False)

In [ ]:
resumen_filtros = []

df_filtrado = datos_2025

def aplicar_filtro(df, condicion, nombre):
    antes = df.count()

    nuevo = df.filter(condicion)

    despues = nuevo.count()

    resumen_filtros.append({
        "filtro": nombre,
        "antes": antes,
        "despues": despues,
        "excluidos": antes - despues
    })

    return nuevo

# 1. Edad válida
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("edad").isNotNull() &
    (~F.isnan("edad")) &
    (F.col("edad") >= 15),
    "Edad >= 15 y válida"
)

# 2. Ocupados
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("ocupado") == "1",
    "OCUPADOS = 1"
)

# 3. Asalariados
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("categoria_ocupacional").isin(
        ["1", "2", "3", "4"]
    ),
    "Categoría ocupacional 1-4"
)

# 4. Salario válido y positivo
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("salario_mensual").isNotNull() &
    (~F.isnan("salario_mensual")) &
    (F.col("salario_mensual") > 0),
    "Salario mensual > 0"
)

# 5. Años de antigüedad válidos
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("antiguedad_anios").isNotNull() &
    (~F.isnan("antiguedad_anios")) &
    (F.col("antiguedad_anios") >= 0),
    "Antigüedad en años válida"
)

# 6. Meses válidos
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("antiguedad_meses").isNotNull() &
    (~F.isnan("antiguedad_meses")) &
    (F.col("antiguedad_meses") >= 0) &
    (F.col("antiguedad_meses") <= 11) &
    (F.col("antiguedad_meses") == F.floor("antiguedad_meses")),
    "Meses de antigüedad entre 0 y 11"
)

# 7. Antigüedad <= edad
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("antiguedad").isNotNull() &
    (~F.isnan("antiguedad")) &
    (F.col("antiguedad") >= 0) &
    (F.col("antiguedad") <= F.col("edad")),
    "Antigüedad <= edad"
)

# 8. Horas válidas
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("horas_semanales").isNotNull() &
    (~F.isnan("horas_semanales")) &
    (F.col("horas_semanales") > 0) &
    (F.col("horas_semanales") <= 168),
    "0 < horas semanales <= 168"
)

resumen_filtros_pd = pd.DataFrame(
    resumen_filtros
)

resumen_filtros_pd

print(
    "Registros originales 2025:",
    total_2025
)

print(
    "Registros analíticos finales:",
    df_filtrado.count()
)

datos_2025_filtrados = df_filtrado

datos_2025_filtrados.groupBy(
    "periodo_archivo"
).count().orderBy(
    "periodo_archivo"
).show()

# Ejercicio 1. Carga, armonización y calidad de datos

A partir de este punto se completa la preparación de 2025 y 2026. Los cuatro trimestres de 2025 se mantienen unidos por nombre y los filtros se aplican en un orden fijo para poder contabilizar las exclusiones.


In [ ]:
# Comparación de registros antes y después de los filtros en 2025
antes = (
    datos_2025.groupBy("periodo_archivo")
    .count()
    .withColumnRenamed("count", "antes")
)

despues = (
    datos_2025_filtrados.groupBy("periodo_archivo")
    .count()
    .withColumnRenamed("count", "despues")
)

comparacion_registros = (
    antes.join(despues, on="periodo_archivo", how="left")
    .withColumn("excluidos", F.col("antes") - F.col("despues"))
    .withColumn(
        "porcentaje_excluido",
        F.round(F.col("excluidos") / F.col("antes") * 100, 2)
    )
    .orderBy("periodo_archivo")
)

comparacion_registros.show(truncate=False)


In [ ]:
# Preparación de 2026 con exactamente las mismas reglas analíticas

def filtrar_poblacion(df):
    return (
        df
        .filter(F.col("edad").isNotNull() & (~F.isnan("edad")) & (F.col("edad") >= 15))
        .filter(F.col("ocupado") == "1")
        .filter(F.col("categoria_ocupacional").isin(["1", "2", "3", "4"]))
        .filter(
            F.col("salario_mensual").isNotNull() &
            (~F.isnan("salario_mensual")) &
            (F.col("salario_mensual") > 0)
        )
        .filter(
            F.col("antiguedad_anios").isNotNull() &
            (~F.isnan("antiguedad_anios")) &
            (F.col("antiguedad_anios") >= 0)
        )
        .filter(
            F.col("antiguedad_meses").isNotNull() &
            (~F.isnan("antiguedad_meses")) &
            (F.col("antiguedad_meses") >= 0) &
            (F.col("antiguedad_meses") <= 11) &
            (F.col("antiguedad_meses") == F.floor("antiguedad_meses"))
        )
        .filter(
            F.col("antiguedad").isNotNull() &
            (~F.isnan("antiguedad")) &
            (F.col("antiguedad") >= 0) &
            (F.col("antiguedad") <= F.col("edad"))
        )
        .filter(
            F.col("horas_semanales").isNotNull() &
            (~F.isnan("horas_semanales")) &
            (F.col("horas_semanales") > 0) &
            (F.col("horas_semanales") <= 168)
        )
    )

# Los faltantes categóricos no reconocidos se representan como DESCONOCIDO.
datos_2025_filtrados = (
    datos_2025_filtrados
    .fillna({"nivel_educativo": "DESCONOCIDO", "dominio": "DESCONOCIDO"})
    .cache()
)

datos_2026_filtrados = (
    filtrar_poblacion(spark_t1_2026)
    .fillna({"nivel_educativo": "DESCONOCIDO", "dominio": "DESCONOCIDO"})
    .cache()
)

print("Registros analíticos 2025:", datos_2025_filtrados.count())
print("Registros analíticos 2026 T1:", datos_2026_filtrados.count())


In [ ]:
# Verificación de unicidad de la clave en 2026

duplicados_clave_2026 = (
    spark_t1_2026
    .groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
    .count()
    .filter(F.col("count") > 1)
)

print("Claves duplicadas en 2026 T1:", duplicados_clave_2026.count())
duplicados_clave_2026.show(10, truncate=False)


In [ ]:
# Guardar los conjuntos preparados por separado en Parquet
import os
os.makedirs("datos_procesados", exist_ok=True)

datos_2025_filtrados.write.mode("overwrite").parquet(
    "datos_procesados/eneic_2025"
)

datos_2026_filtrados.write.mode("overwrite").parquet(
    "datos_procesados/eneic_2026T1"
)

print("Parquet de 2025 y 2026 guardados correctamente.")


### Respuestas metodológicas del ejercicio 1

**¿Por qué IV de 2025 no puede apilarse por posición?** El cuarto trimestre contiene más columnas originales que los otros archivos. Por ello se seleccionaron y homologaron las variables requeridas y se utilizó `unionByName`, evitando depender de la posición física de las columnas.

**Ausencia por no correspondencia vs. respuesta no registrada.** Una pregunta puede no aplicar por el flujo del cuestionario; esto es distinto de una pregunta aplicable cuya respuesta no fue registrada. Ambos casos pueden aparecer como faltantes, pero no tienen el mismo significado.

**Observaciones repetidas entre períodos.** La ENEIC tiene diseño longitudinal con rotación. Una persona observada en dos períodos representa dos observaciones temporales y no debe eliminarse automáticamente como duplicado.

**Alcance de la base filtrada.** Los resultados describen únicamente los registros que cumplen los criterios analíticos y el análisis principal es no ponderado. Por ello no representan a todos los trabajadores del país. `FACTOR` se conserva para documentar el diseño muestral y sería necesario en estimaciones poblacionales.


# Ejercicio 2. Estadística descriptiva y análisis exploratorio


In [ ]:
# Estadísticos descriptivos sobre todos los registros elegibles de 2025
variables_numericas_eda = [
    "salario_mensual", "edad", "antiguedad", "horas_semanales"
]

estadisticos = []

for v in variables_numericas_eda:
    fila = datos_2025_filtrados.select(
        F.lit(v).alias("variable"),
        F.count(F.col(v)).alias("n"),
        F.avg(v).alias("media"),
        F.expr(f"percentile_approx({v}, 0.5, 10000)").alias("mediana"),
        F.stddev(v).alias("desv_estandar"),
        F.min(v).alias("minimo"),
        F.expr(f"percentile_approx({v}, 0.25, 10000)").alias("p25"),
        F.expr(f"percentile_approx({v}, 0.75, 10000)").alias("p75"),
        F.expr(f"percentile_approx({v}, 0.95, 10000)").alias("p95"),
        F.max(v).alias("maximo")
    )
    estadisticos.append(fila)

tabla_estadisticos = estadisticos[0]
for tabla in estadisticos[1:]:
    tabla_estadisticos = tabla_estadisticos.unionByName(tabla)

tabla_estadisticos.show(truncate=False)


In [ ]:
# Distribución de registros por categoría ocupacional, nivel educativo y dominio
for variable in ["categoria_ocupacional", "nivel_educativo", "dominio"]:
    print(f"\nDistribución de {variable}")
    (
        datos_2025_filtrados
        .groupBy(variable)
        .count()
        .withColumn(
            "porcentaje",
            F.round(F.col("count") / F.lit(datos_2025_filtrados.count()) * 100, 2)
        )
        .orderBy(F.desc("count"))
        .show(50, truncate=False)
    )


In [ ]:
# Gráficas de las variables categóricas usando únicamente tablas agregadas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, variable, titulo in zip(
    axes,
    ["categoria_ocupacional", "nivel_educativo", "dominio"],
    ["Categoría ocupacional", "Nivel educativo", "Dominio"]
):
    tabla = (
        datos_2025_filtrados.groupBy(variable).count()
        .orderBy(F.desc("count"))
        .toPandas()
    )
    ax.bar(tabla[variable].astype(str), tabla["count"])
    ax.set_title(titulo)
    ax.set_xlabel(variable)
    ax.set_ylabel("Registros")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Distribución del salario. Se usa una muestra solo para dibujar; las estadísticas anteriores usan todos los registros.

muestra_salario = (
    datos_2025_filtrados
    .select("salario_mensual")
    .sample(withReplacement=False, fraction=min(1.0, 10000 / datos_2025_filtrados.count()), seed=42)
    .limit(10000)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(muestra_salario["salario_mensual"], bins=50)
axes[0].set_title("Distribución del salario mensual")
axes[0].set_xlabel("Salario mensual (Q)")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(muestra_salario["salario_mensual"], bins=50)
axes[1].set_xscale("log")
axes[1].set_title("Distribución del salario mensual (eje X logarítmico)")
axes[1].set_xlabel("Salario mensual (Q, escala log)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()


In [ ]:
# Salario mediano por nivel educativo y categoría ocupacional

mediana_educacion = (
    datos_2025_filtrados
    .groupBy("nivel_educativo")
    .agg(
        F.count("*").alias("n"),
        F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias("salario_mediano")
    )
    .orderBy(F.desc("salario_mediano"))
)

mediana_ocupacion = (
    datos_2025_filtrados
    .groupBy("categoria_ocupacional")
    .agg(
        F.count("*").alias("n"),
        F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias("salario_mediano")
    )
    .orderBy(F.desc("salario_mediano"))
)

print("Salario mediano por nivel educativo")
mediana_educacion.show(50, truncate=False)
print("Salario mediano por categoría ocupacional")
mediana_ocupacion.show(20, truncate=False)


In [ ]:
# Tamaño de muestra y salario mediano por trimestre
resumen_trimestre = (
    datos_2025_filtrados
    .groupBy("periodo_archivo")
    .agg(
        F.count("*").alias("n"),
        F.avg("salario_mensual").alias("salario_medio"),
        F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias("salario_mediano")
    )
    .orderBy("periodo_archivo")
)

resumen_trimestre.show(truncate=False)

trimestre_pd = resumen_trimestre.toPandas()
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(trimestre_pd["periodo_archivo"], trimestre_pd["n"])
ax1.set_xlabel("Trimestre")
ax1.set_ylabel("Registros analíticos")
ax1.set_title("Tamaño de muestra analítica por trimestre")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(trimestre_pd["periodo_archivo"], trimestre_pd["salario_mediano"], marker="o")
plt.title("Salario mediano por trimestre")
plt.xlabel("Trimestre")
plt.ylabel("Salario mediano (Q)")
plt.tight_layout()
plt.show()


### Interpretación del ejercicio 2

Después de ejecutar las celdas anteriores, documente en Markdown: (1) qué categorías concentran más registros; (2) si la distribución salarial es simétrica o asimétrica; (3) la diferencia entre media y mediana; (4) cómo cambia el salario mediano según educación y categoría ocupacional; y (5) cómo cambian el tamaño de la muestra y el salario mediano entre trimestres. No elimine salarios extremos únicamente por ser altos o bajos.


# Ejercicio 3. Relaciones entre variables numéricas


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

variables_correlacion = [
    "salario_mensual", "edad", "antiguedad", "horas_semanales"
]

assembler_corr = VectorAssembler(
    inputCols=variables_correlacion,
    outputCol="features_corr",
    handleInvalid="skip"
)

datos_corr = assembler_corr.transform(datos_2025_filtrados).select("features_corr")

matriz = Correlation.corr(datos_corr, "features_corr", "pearson").head()[0]
matriz_np = matriz.toArray()

corr_pd = pd.DataFrame(
    matriz_np,
    index=variables_correlacion,
    columns=variables_correlacion
)

display(corr_pd.round(3))

plt.figure(figsize=(7, 6))
plt.imshow(corr_pd, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(label="Correlación de Pearson")
plt.xticks(range(len(variables_correlacion)), variables_correlacion, rotation=45, ha="right")
plt.yticks(range(len(variables_correlacion)), variables_correlacion)

for i in range(len(variables_correlacion)):
    for j in range(len(variables_correlacion)):
        plt.text(j, i, f"{corr_pd.iloc[i, j]:.2f}", ha="center", va="center")

plt.title("Matriz de correlaciones - población analítica 2025")
plt.tight_layout()
plt.show()


### Interpretación del ejercicio 3

Use la matriz anterior para identificar cuál de edad, antigüedad y horas semanales presenta la mayor asociación lineal con el salario. Discuta también la correlación entre edad y antigüedad. Recuerde que correlación describe asociación lineal y no demuestra causalidad.


# Ejercicio 4. Segmentación de perfiles mediante KMeans

Se comparan dos alternativas: una segmentación basada únicamente en edad, antigüedad y horas habituales, y otra que además incorpora salario mensual. Esto permite evaluar empíricamente si incluir salario mejora la separación de los perfiles. Todas las variables se estandarizan antes de KMeans.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

escenarios_cluster = {
    "sin_salario": ["edad", "antiguedad", "horas_semanales"],
    "con_salario": ["edad", "antiguedad", "horas_semanales", "salario_mensual"]
}

resultados_k = []
modelos_k = {}
datos_clusterizados = {}

evaluador = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features_scaled",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

for escenario, variables in escenarios_cluster.items():
    assembler = VectorAssembler(
        inputCols=variables,
        outputCol="features_raw",
        handleInvalid="skip"
    )
    base = assembler.transform(datos_2025_filtrados)

    scaler = StandardScaler(
        inputCol="features_raw",
        outputCol="features_scaled",
        withStd=True,
        withMean=True
    )
    scaler_model = scaler.fit(base)
    base_scaled = scaler_model.transform(base).cache()

    for k in [2, 3, 4, 5]:
        kmeans = KMeans(
            k=k,
            seed=42,
            featuresCol="features_scaled",
            predictionCol="prediction"
        )
        modelo = kmeans.fit(base_scaled)
        pred = modelo.transform(base_scaled)
        silhouette = evaluador.evaluate(pred)

        resultados_k.append({
            "escenario": escenario,
            "k": k,
            "silhouette": silhouette
        })
        modelos_k[(escenario, k)] = modelo
        datos_clusterizados[(escenario, k)] = pred

resultados_k_pd = pd.DataFrame(resultados_k).sort_values(
    ["escenario", "k"]
)
resultados_k_pd


In [ ]:
# Visualización del criterio de selección de K
plt.figure(figsize=(8, 5))
for escenario in resultados_k_pd["escenario"].unique():
    sub = resultados_k_pd[resultados_k_pd["escenario"] == escenario]
    plt.plot(sub["k"], sub["silhouette"], marker="o", label=escenario)

plt.xlabel("Número de clusters (K)")
plt.ylabel("Silhouette")
plt.title("Selección de K mediante silhouette")
plt.xticks([2, 3, 4, 5])
plt.legend()
plt.tight_layout()
plt.show()

mejor = resultados_k_pd.loc[resultados_k_pd["silhouette"].idxmax()]
mejor_escenario = mejor["escenario"]
mejor_k = int(mejor["k"])

print("Mejor escenario:", mejor_escenario)
print("Mejor K:", mejor_k)
print("Silhouette:", round(mejor["silhouette"], 4))


In [ ]:
# Caracterización de los clusters seleccionados
from pyspark.sql.window import Window

pred_final = datos_clusterizados[(mejor_escenario, mejor_k)]

perfil_clusters = (
    pred_final
    .groupBy("prediction")
    .agg(
        F.count("*").alias("n"),
        F.avg("edad").alias("edad_media"),
        F.avg("antiguedad").alias("antiguedad_media"),
        F.avg("horas_semanales").alias("horas_media"),
        F.avg("salario_mensual").alias("salario_medio"),
        F.expr("percentile_approx(salario_mensual, 0.5, 10000)").alias("salario_mediano")
    )
    .orderBy("prediction")
)

perfil_clusters.show(truncate=False)

# Distribuciones categóricas por cluster para ayudar a describir los perfiles
for variable in ["nivel_educativo", "categoria_ocupacional", "dominio"]:
    print(f"\n{variable} por cluster")
    (
        pred_final
        .groupBy("prediction", variable)
        .count()
        .withColumn(
            "total_cluster",
            F.sum("count").over(Window.partitionBy("prediction"))
        )
        .withColumn(
            "porcentaje_cluster",
            F.round(F.col("count") / F.col("total_cluster") * 100, 2)
        )
        .orderBy("prediction", F.desc("count"))
        .show(100, truncate=False)
    )


### Interpretación del ejercicio 4

El K final se selecciona comparando el coeficiente silhouette para K = 2, 3, 4 y 5 en los escenarios con y sin salario. La decisión final debe considerar tanto la separación cuantitativa como la interpretabilidad de los perfiles.

Con la tabla `perfil_clusters`, asigne una descripción sustantiva a cada cluster utilizando edad, antigüedad, jornada habitual y salario. Las tablas categóricas pueden complementar la descripción con nivel educativo, categoría ocupacional y dominio. Los clusters describen grupos de registros similares; no implican categorías naturales ni relaciones causales.
